In [3]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression, NaiveBayes, LinearSVC
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import pandas as pd

spark = SparkSession.builder \
    .appName("AmazonSentiment") \
    .config("spark.driver.memory", "6g") \
    .config("spark.executor.memory", "6g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
train = spark.createDataFrame(
    pd.read_csv('/Users/beethoven/BigData/AmazonReview/data/train.csv')
    [['clean_text', 'sentiment', 'ProductId', 'UserId', 'Id', 'Time']].dropna()
)

val = spark.createDataFrame(
    pd.read_csv('/Users/beethoven/BigData/AmazonReview/data/val.csv')
    [['clean_text', 'sentiment', 'ProductId', 'UserId', 'Id', 'Time']].dropna()
)

test = spark.createDataFrame(
    pd.read_csv('/Users/beethoven/BigData/AmazonReview/data/test.csv')
    [['clean_text', 'sentiment', 'ProductId', 'UserId', 'Id', 'Time']].dropna()
)

print(f"Train: {train.count()} | Val: {val.count()} | Test: {test.count()}")

In [ ]:
tokenizer = Tokenizer(inputCol="clean_text", outputCol="words")

remover = StopWordsRemover(inputCol="words", outputCol="filtered")

hashingTF = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=50000)

idf = IDF(inputCol="rawFeatures", outputCol="features", minDocFreq=5)

indexer = StringIndexer(inputCol="sentiment", outputCol="label")

In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

results = {}
trained_models = {}

# --- Logistic Regression ---
print("Training Logistic Regression...")
lr = LogisticRegression(maxIter=100, regParam=0.01, elasticNetParam=0.0, family="multinomial")
pipeline_lr = Pipeline(stages=[tokenizer, remover, hashingTF, idf, indexer, lr])
model_lr = pipeline_lr.fit(train)
f1_lr = evaluator.evaluate(model_lr.transform(val))
results['LogisticRegression'] = f1_lr
trained_models['LogisticRegression'] = model_lr
print(f"LR Val F1: {f1_lr:.4f}")

# --- Naive Bayes ---
print("Training Naive Bayes...")
nb = NaiveBayes(smoothing=1.0, modelType="multinomial")
pipeline_nb = Pipeline(stages=[tokenizer, remover, hashingTF, idf, indexer, nb])
model_nb = pipeline_nb.fit(train)
f1_nb = evaluator.evaluate(model_nb.transform(val))
results['NaiveBayes'] = f1_nb
trained_models['NaiveBayes'] = model_nb
print(f"NB Val F1: {f1_nb:.4f}")

# --- Linear SVC ---
print("Training LinearSVC...")
svc = LinearSVC(maxIter=100, regParam=0.01)
pipeline_svc = Pipeline(stages=[tokenizer, remover, hashingTF, idf, indexer, svc])
model_svc = pipeline_svc.fit(train)
f1_svc = evaluator.evaluate(model_svc.transform(val))
results['LinearSVC'] = f1_svc
trained_models['LinearSVC'] = model_svc
print(f"SVC Val F1: {f1_svc:.4f}")

print(f"\nAll results: {results}")

In [ ]:
best_name = max(results, key=results.get)
best_model = trained_models[best_name]

print(f"Best model: {best_name}")
print(f"Val F1: {results[best_name]:.4f}")

# Evaluate on test set
test_preds = best_model.transform(test)
test_f1 = evaluator.evaluate(test_preds)
print(f"Test F1: {test_f1:.4f}")

# Confusion matrix breakdown
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

for metric in ['f1', 'accuracy', 'weightedPrecision', 'weightedRecall']:
    evaluator.setMetricName(metric)
    score = evaluator.evaluate(test_preds)
    print(f"{metric}: {score:.4f}")

In [ ]:
model_path = '/Users/beethoven/BigData/AmazonReview/model/best_sentiment_model'
best_model.write().overwrite().save(model_path)
print(f"Model saved to {model_path}")
print(f"Best model was: {best_name}")

In [ ]:
# You need this later in streaming to convert numeric predictions back to sentiment strings
indexer_model = best_model.stages[4]  # StringIndexer is stage index 4
labels = indexer_model.labels
print(f"Label mapping: {list(enumerate(labels))}")

import json
with open('/Users/beethoven/BigData/AmazonReview/model/label_mapping.json', 'w') as f:
    json.dump({str(i): label for i, label in enumerate(labels)}, f)
print("Label mapping saved.")